# Assignment 1

Please select one city in US and generate the NDVI map and the vegetation cover map at the census tract level.
Collect the city limit file of the city.
Download the NAIP tiles and mosaic them.
Mask out those pixel out of the bound.
Calculate the NDVI and vegetation cover map
Generate the census tract level map of NDVI and vegetation cover

## City-level NDVI and vegetation workflow

This notebook uses Miami, Florida as the selected U.S. city. It combines NAIP four-band aerial imagery with a Miami city-limit boundary and census-tract polygons to produce city-level and tract-level vegetation results.

### Assignment workflow

1. Collect the Miami city-limit file and confirm it is in WGS 84, EPSG:4326 for the search area.
2. Look for existing NAIP GeoTIFFs in `cir-naip`; if none are available, search the Planetary Computer and download the NAIP tiles.
3. Mosaic the selected tiles into one georeferenced raster using a memory-limited disk-based merge.
4. Reproject the EPSG:4326 city boundary into the mosaic CRS and mask pixels outside Miami.
5. Split the masked raster into manageable windows, including partial edge tiles.
6. Calculate NDVI from the NAIP red band 1 and near-infrared band 4.
7. Classify vegetation using the assignment threshold `NDVI > 0.1`.
8. Reproject the census-tract layer into the raster CRS, calculate tract mean NDVI and vegetation percentage, and map the results.

### Interpretation notes

NDVI is calculated as $(NIR - Red) / (NIR + Red)$. The vegetation map is a threshold classification, not a tree-canopy estimate. A native pixel is vegetation when its NDVI is greater than `NDVI_THRESHOLD`, which is set to `0.1` below.

NoData pixels are kept separate from valid zero NDVI and valid non-vegetation pixels. Areas outside Miami are masked during raster processing and displayed in gray on the census-tract maps. Tracts that cross the city boundary are clipped before their values are drawn, so color does not extend outside Miami.

### Running the notebook

Run the cells from top to bottom. The notebook first looks for existing GeoTIFFs in `cir-naip`; if none are found, it searches the Planetary Computer and downloads the selected NAIP tiles automatically. Edit `INPUT_ROOT` if the source folder moves. Generated GeoTIFFs, GeoPackage summaries, and PNG maps are written under `city-workflow/data/city-products/`.

Required packages include rasterio, numpy, geopandas, fiona, shapely, matplotlib, and, for automatic downloads, pystac-client and planetary-computer.


In [ ]:
# CHANGED — local input configuration; no catalog connection needed by default
# --- standard library ---------------------------------------------------
import os
import os.path
import glob
import urllib.request                      # downloads a URL to a local file

# --- searching and signing the Planetary Computer catalog ---------------
from pathlib import Path
USE_LOCAL_FILES = True
INPUT_ROOT = Path('/Users/scorper/Documents/UPenn/AI Urban Sustainability/Leo_Leung_Assignment_1-Lab-3')
# All generated files stay under the notebook working directory.
WORK_ROOT = Path.cwd() / 'city-workflow'
if not USE_LOCAL_FILES:
    import pystac_client
    import planetary_computer as pc

# --- raster and vector I/O ----------------------------------------------
import rasterio as rio                     # read/write GeoTIFFs
import fiona                               # read/write vector files (shapefiles)
import geopandas as gpd                    # vector data as a table (used for a quick map)

# --- geometry and arrays -------------------------------------------------
import numpy as np
from shapely.geometry import shape, box, mapping   # build/convert geometries

# --- plotting -------------------------------------------------------------
from matplotlib import pyplot as plt

# --- Raster ---------------------------------------------------------------
from rasterio.plot import show
from rasterio.mask import mask
from rasterio.windows import get_data_window
from rasterio.windows import Window
from rasterio.merge import merge
# optional - turn off warnings
import warnings

import glob
from shapely.geometry import Polygon
from shapely.geometry import shape

print("Input folder:", INPUT_ROOT)


In [ ]:
# CHANGED — catalog/download mode is optional
if not USE_LOCAL_FILES:
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1/",
        modifier=pc.sign_inplace,   # every asset URL comes back ready to download
    )
    
    catalog


In [ ]:
# CHANGED — use the supplied project boundary
shpfile = str(INPUT_ROOT / 'data' / 'Miami_boundary_4326.shp')

# fiona opens a vector file as a stream of features. Using `with` closes it for us.
with fiona.open(shpfile) as lyr:
    print("CRS of the input file:", lyr.crs.to_string())   # must be EPSG:4326
    print("Number of features   :", len(lyr))

    # Convert every feature to a shapely geometry. Keeping them in a list (rather
    # than merging them) means this works for a boundary made of several polygons.
    city_parts = [shape(feat['geometry']) for feat in lyr]

    # .bounds gives the extent of the whole layer: (minx, miny, maxx, maxy)
    left, bottom, right, top = lyr.bounds

print("\nBounding box (lon/lat):", (left, bottom, right, top))

area_of_interest = mapping(box(left, bottom, right, top))
area_of_interest


In [ ]:
# Sanity check: draw the city outline inside its bounding box.
# Always look at your AOI before firing off an 8 GB download.
fig, ax = plt.subplots(figsize=(6, 6))

gpd.read_file(shpfile).plot(ax=ax, color="lightblue", edgecolor="black", linewidth=0.6)
gpd.GeoSeries([box(left, bottom, right, top)], crs="EPSG:4326").boundary.plot(
    ax=ax, color="red", linestyle="--", linewidth=1
)

ax.set_title("Area of interest: city limit (blue) and its bounding box (red)")
plt.show()

In [ ]:
# CHANGED — catalog/download mode is optional
if not USE_LOCAL_FILES:
    # CHANGED — guard against an empty imagery search
    startDate = '2019-01-01'
    endDate   = '2020-12-31'
    range_time = f"{startDate}/{endDate}"     # STAC wants a single "START/END" string
    
    search = catalog.search(
        collections=["naip"],          # which dataset
        intersects= area_of_interest,   # where  (GeoJSON geometry, EPSG:4326)
        datetime=range_time,           # when
    )
    
    # .item_collection() actually sends the request and pages through all the results.
    # (Older tutorials call .get_items() -- that name is deprecated.)
    items_tiles = search.item_collection()
    
    print(f"{len(items_tiles)} NAIP tiles found for {range_time}")
    if not items_tiles:
        raise ValueError("No NAIP tiles found. Check city extent and date range.")


In [ ]:
# CHANGED — catalog/download mode is optional
if not USE_LOCAL_FILES:
    # Look at a single Item to see what the catalog knows about it
    item = items_tiles[0]
    
    print("id       :", item.id)
    print("date     :", item.datetime.date())
    print("state    :", item.properties.get("naip:state"))
    print("gsd      :", item.properties.get("gsd"), "m per pixel")
    print("assets   :", list(item.assets))
    print("\nimage href (signed, truncated):")
    print(item.assets["image"].href[:120], "...")
    # How much data are we about to pull down?
    # The catalog does not report file size, so use ~165 MB per NAIP tile as a rule of thumb.
    MB_PER_TILE = 165
    print(f"{len(items_tiles)} tiles x ~{MB_PER_TILE} MB "
          f"= roughly {len(items_tiles) * MB_PER_TILE / 1024:.1f} GB")
    
    # Which acquisition dates are in the result? NAIP tiles for one city are often
    # flown across several days, which is why a mosaic can show visible seams.
    dates = sorted({i.datetime.date().isoformat() for i in items_tiles})
    print("\nacquisition dates:", dates)


In [ ]:
# CHANGED — catalog/download mode is optional
if not USE_LOCAL_FILES:
    # How much data are we about to pull down?
    # The catalog does not report file size, so use ~165 MB per NAIP tile as a rule of thumb.
    MB_PER_TILE = 165
    print(f"{len(items_tiles)} tiles x ~{MB_PER_TILE} MB "
          f"= roughly {len(items_tiles) * MB_PER_TILE / 1024:.1f} GB")
    
    # Which acquisition dates are in the result? NAIP tiles for one city are often
    # flown across several days, which is why a mosaic can show visible seams.
    dates = sorted({i.datetime.date().isoformat() for i in items_tiles})
    print("\nacquisition dates:", dates)


In [ ]:
# CHANGED — refresh the catalog and sign each missing tile immediately before download
import pystac_client
import planetary_computer as pc

outfolder = 'cir-naip'
os.makedirs(outfolder, exist_ok=True)   # exist_ok=True: no error if it is already there

MAX_TILES = None      # set to e.g. 2 to try the workflow out without an 8 GB download
catalog = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1/')
search = catalog.search(
    collections=['naip'],
    intersects=area_of_interest,
    datetime='2019-01-01/2020-12-31',
)
items_tiles = search.item_collection()
to_download = items_tiles[:MAX_TILES] if MAX_TILES else items_tiles
if not to_download:
    raise ValueError('The NAIP search returned no tiles to download.')

local_count = len(glob.glob(os.path.join(outfolder, '*.tif')))
print(f'Found {local_count} local NAIP tiles; catalog selected {len(to_download)} tiles')
for n, item in enumerate(to_download, start=1):
    outfilename = os.path.join(outfolder, item.id + '.tif')

    # Already have it? Move on. This makes the cell safe to re-run.
    if os.path.exists(outfilename):
        print(f'[{n}/{len(to_download)}] {item.id} -- already on disk, skipping')
        continue

    # Sign this freshly queried item so no expired SAS token is reused.
    pc.sign_inplace(item)
    href = item.assets['image'].href

    print(f'[{n}/{len(to_download)}] downloading {item.id} ...')
    urllib.request.urlretrieve(href, outfilename)

local_naip_files = sorted(glob.glob(os.path.join(outfolder, '*.tif')))
print(f'Available NAIP tiles: {len(local_naip_files)}')
if not local_naip_files:
    raise FileNotFoundError('No NAIP GeoTIFFs are available after the download step.')


# 1. Configure the city workflow

This section defines the study area, input imagery, output locations, and analysis parameters. `USE_LOCAL_FILES=True` makes the notebook reproducible with the supplied Miami tiles. Set it to `False` only when you want to query and download imagery from the Planetary Computer.

The source NAIP imagery is four-band data in the order red, green, blue, and near-infrared. `CITY_RES_M` controls the resolution of the city-level products, while `NDVI_THRESHOLD` controls which native pixels count as vegetation. The notebook writes intermediate tiles to `city-workflow/data/` and final products to `city-workflow/data/city-products/`.

Run the notebook from top to bottom so that each later stage uses the CRS, manifests, and file paths created here.


In [ ]:
# CHANGED — city workflow
from pathlib import Path
from contextlib import ExitStack
from rasterio.features import geometry_mask, geometry_window
from rasterio.enums import Resampling
from rasterio.warp import reproject, aligned_target
from rasterio.plot import plotting_extent
import inspect

# Rasterio >=1.4 is required for memory-limited, on-disk merging.
if 'mem_limit' not in inspect.signature(merge).parameters:
    raise RuntimeError('Install rasterio >=1.4 and restart the kernel.')
CITY_NAME = 'Miami'
BOUNDARY_FILE = shpfile  # original EPSG:4326 boundary; reproject automatically below
CITY_RES_M = 10.0       # larger pixels = lower spatial resolution; try 20 or 30
NDVI_THRESHOLD = 0.1    # vegetation if native-resolution NDVI > this value
RED_BAND, NIR_BAND = 1, 4  # NAIP order: red, green, blue, NIR
TILE_SIZE = 2000
FLOAT_NODATA = -9999.0
VEG_NODATA = 255        # 0 remains valid non-vegetation; 1 = vegetation
DATA = WORK_ROOT / 'data'
PRODUCTS = DATA / 'city-products'
PRODUCTS.mkdir(parents=True, exist_ok=True)
# Use only this search/download selection, never stale files from another city.
naip_files = local_naip_files[:MAX_TILES] if MAX_TILES else local_naip_files
if not naip_files or any(not Path(p).is_file() for p in naip_files):
    raise FileNotFoundError('Download the selected NAIP tiles before continuing.')
if MAX_TILES is not None:
    warnings.warn('MAX_TILES is set: outputs may cover only part of the city.')
MOSAIC_FILE = DATA / 'mosaicked-naip.tif'
MASKED_FILE = DATA / 'masked-city-naip.tif'


# Look at a tile

Check the CRS, bounds, resolution and bands before processing. NAIP commonly uses a projected UTM CRS with metre units; the mosaic cell validates the actual CRS.


In [ ]:
# CHANGED — city workflow
tiffile = naip_files[0]
dataset = rio.open(tiffile)
print('Opening:', tiffile)
print('CRS:', dataset.crs, 'Size:', dataset.shape, 'Bands:', dataset.count)
print('Resolution:', dataset.res, 'Bounds:', dataset.bounds)


### 8.1 Reading at reduced resolution
A NAIP tile is about 7,500 x 6,000 pixels. Reading all four bands at full size takes ~180 MB of RAM per tile, and matplotlib cannot show that many pixels on screen anyway.

out_shape tells rasterio to resample while reading, so only the smaller array is ever built. Get in the habit — it is the difference between a plot that appears instantly and one that fills your memory.

In [ ]:
dsfactor = 10                              # read 1 pixel in every 10
h = dataset.height // dsfactor
w = dataset.width  // dsfactor
print(f"Resampling {dataset.width}x{dataset.height} -> {w}x{h} for display")

# rasterio band indices start at 1
red   = dataset.read(1, out_shape=(h, w))
green = dataset.read(2, out_shape=(h, w))
blue  = dataset.read(3, out_shape=(h, w))
nir   = dataset.read(4, out_shape=(h, w))

for name, band in [("red", red), ("green", green), ("blue", blue), ("NIR", nir)]:
    print(f"{name:>5}: min={band.min():3d}  max={band.max():3d}  mean={band.mean():6.1f}")

### 8.2 Natural colour vs colour-infrared


np.dstack stacks the three 2-D arrays into one (height, width, 3) array, which is the shape matplotlib expects for a colour image.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Natural colour: R, G, B -> screen R, G, B
natural = np.dstack((red, green, blue))
axes[0].imshow(natural)
axes[0].set_title("Natural colour  (bands 1, 2, 3)")

# Colour-infrared: NIR, R, G -> screen R, G, B
cir = np.dstack((nir, red, green))
axes[1].imshow(cir)
axes[1].set_title("Colour-infrared  (bands 4, 1, 2) -- vegetation appears red")

for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# A single band on its own is just a grey-scale array of numbers.
# NIR is the interesting one: bright = vegetation, dark = water and asphalt.
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(nir, cmap='gray')
ax.set_title("Near-infrared band (band 4)")
ax.set_axis_off()
plt.colorbar(im, ax=ax, shrink=0.7, label="reflectance (0-255)")
plt.show()

dataset.close()   # release the file handle when you are finished with it

# Raster data operations
Raster data is an imporant format of spatial data. In this tutorial, we are going to using Python to manipulate the raster data format. We will work with RasterIO and Numpy for the raster data processing. RasterIO is a modern library to work with geospatial data in a gridded format. It excels at providing an easy way to read/write raster data and access individual bands and pixels as numpy arrays.

Rasterio is built on top of the popular GDAL (Geospatial Data Abstraction Library). GDAL is written in C++ so the Python API provided by GDAL is not very intuitive for Python users. RaserIO aims to make it easy for Python users to use the underlying GDAL library in an intuitive way.

In this section, we will read and write raster data using rasterio.

### Read the raster data and get the metadata of the raster data
After running the notebook of naip-downloader.ipynb, you will download the NAIP tiles for the selected city. Then we can check the data and explore how to use Python to do analyses on the raster data. If you have difficulty to download the NAIP tiles, I provide one for you to download.

In [ ]:
# CHANGED — city workflow
naip_file = naip_files[0]
with rio.open(naip_file) as naip_dataset:
    naip_bounds = naip_dataset.bounds
    naip_metadata = naip_dataset.meta.copy()
naip_bounds


In [ ]:
# CHANGED — city workflow
naip_metadata


Now we can read the geotiff image for different bands

In [ ]:
# CHANGED — city workflow
with rio.open(naip_file) as src:
    preview = src.read([1, 2, 3], out_shape=(3, max(1, src.height//10), max(1, src.width//10)))
plt.figure(figsize=(10, 10))
plt.imshow(np.moveaxis(preview, 0, -1))
plt.title('NAIP tile — RGB preview')
plt.axis('off')
plt.show()


# City NDVI and vegetation workflow (CHANGED)
1. Mosaic downloaded NAIP tiles at native resolution on disk.
2. Reproject the city boundary and mask the mosaic in windows.
3. Split into processing tiles, including partial edge tiles.
4. Calculate native NDVI and binary vegetation per tile using red band 1 and NIR band 4.
5. **NEW:** assemble processed tiles into single city rasters.
6. **NEW:** average native NDVI and binary vegetation onto one lower-resolution grid, then map and export.

Vegetation is an NDVI threshold classification, not a tree-canopy classification.
The threshold is adjustable; 0.2 is an illustrative starting value requiring local validation.


In [ ]:
print('Working directory:', Path.cwd())

In [ ]:
# Deterministic first-valid-pixel overlap rule; no averaging of acquisition dates.
# Keep a coherent acquisition period in the STAC search to avoid temporal seams.
with ExitStack() as stack:
    sources = [stack.enter_context(rio.open(p)) for p in naip_files]
    first = sources[0]
    if first.crs is None or first.count < 4:
        raise ValueError('NAIP imagery needs a CRS and four RGB-NIR bands.')
    if not first.crs.is_projected or abs(first.crs.linear_units_factor[1] - 1) > 1e-6:
        raise ValueError('Use NAIP tiles in a projected CRS with metre units.')
    for src in sources:
        if (src.crs != first.crs or src.count != first.count or
                src.dtypes != first.dtypes or src.res != first.res):
            raise ValueError('Tiles must share CRS, band count, dtype and resolution; reproject first.')
    # 0 is the NAIP background sentinel; NDVI validity is checked again below.
    merge(sources, dst_path=str(MOSAIC_FILE), nodata=0, method='first',
          mem_limit=128, dst_kwds=dict(compress='lzw', tiled=True, blockxsize=256, blockysize=256, BIGTIFF='IF_SAFER'))
# Explicitly preserve RGB-NIR: the fourth band is not an alpha mask.
with rio.open(MOSAIC_FILE, 'r+') as dst:
    dst.colorinterp = (rio.enums.ColorInterp.red, rio.enums.ColorInterp.green,
                       rio.enums.ColorInterp.blue, rio.enums.ColorInterp.undefined)
print('Mosaicked', len(naip_files), 'tiles:', MOSAIC_FILE)


### 3. Mask the mosaic to the city boundary

The mosaic may cover more area than Miami. The boundary is reprojected into the mosaic CRS before it is used, which makes the polygon and pixels line up correctly even when the source boundary is stored in longitude/latitude.

The code reads and writes the mosaic in windows instead of loading the complete raster into memory. For each window, `geometry_mask` identifies pixels whose centers are inside Miami. Pixels outside the boundary are written as NoData, including pixels in holes in a multipart boundary. This mask becomes the spatial limit for every later NDVI and vegetation product.


In [ ]:
# CHANGED — city workflow
city_boundary = gpd.read_file(BOUNDARY_FILE)
if city_boundary.crs is None or city_boundary.empty:
    raise ValueError('City boundary must be nonempty and have a CRS.')
with rio.open(MOSAIC_FILE) as src:
    city_projected = city_boundary.to_crs(src.crs)
    city_geometries = [mapping(g) for g in city_projected.geometry if g is not None and not g.is_empty]
    crop_window = geometry_window(src, city_geometries)
    profile = src.profile.copy()
    profile.update(width=int(crop_window.width), height=int(crop_window.height),
                   transform=src.window_transform(crop_window), nodata=0)
    with rio.open(MASKED_FILE, 'w', **profile) as dst:
        dst.colorinterp = src.colorinterp
        for _, window in dst.block_windows(1):
            source_window = Window(window.col_off + crop_window.col_off,
                                   window.row_off + crop_window.row_off,
                                   window.width, window.height)
            pixels = src.read(window=source_window)
            inside = geometry_mask(city_geometries,
                out_shape=(int(window.height), int(window.width)),
                transform=dst.window_transform(window), invert=True)
            valid = inside & (src.dataset_mask(window=source_window) > 0)
            pixels[:, ~valid] = 0
            dst.write(pixels, window=window)
            dst.write_mask(valid.astype('uint8') * 255, window=window)
print('Masked city imagery:', MASKED_FILE)



### 4. Split the masked raster into processing tiles

A full city mosaic can be too large to process comfortably in one array. The masked raster is therefore divided into fixed-size windows. The final windows at the right and bottom edges are kept even when they are smaller than `TILE_SIZE`; dropping them would create gaps in the city output.

Each window retains its georeferencing and mask. The explicit `processing_tiles` manifest is used later so that stale files from an earlier run cannot accidentally enter the calculation.


In [ ]:
# CHANGED — city workflow
tile_folder = DATA / 'naip-tiles'
tile_folder.mkdir(exist_ok=True)
processing_tiles = []  # explicit manifest excludes stale files on reruns
with rio.open(MASKED_FILE) as src:
    for row in range(0, src.height, TILE_SIZE):
        for col in range(0, src.width, TILE_SIZE):
            window = Window(col, row, min(TILE_SIZE, src.width-col), min(TILE_SIZE, src.height-row))
            path = tile_folder / f'row{row//TILE_SIZE}-col{col//TILE_SIZE}.tif'
            profile = src.profile.copy()
            profile.update(height=int(window.height), width=int(window.width),
                           transform=src.window_transform(window))
            with rio.open(path, 'w', **profile) as dst:
                dst.colorinterp = src.colorinterp
                dst.write(src.read(window=window))
                dst.write_mask(src.dataset_mask(window=window))
            processing_tiles.append(path)
print('Processing tiles (including edges):', len(processing_tiles))


### 5. Calculate native NDVI and vegetation

For every processing tile, the code reads the red and NIR bands and computes NDVI using:

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

Pixels are valid only when both bands have data, the denominator is positive, and the result is finite. Invalid pixels receive the NDVI NoData value. Following the assignment hint, valid pixels are classified as vegetation when `NDVI > 0.1`; the binary vegetation raster uses `0` for valid non-vegetation, `1` for vegetation, and `255` for NoData.

Keeping NoData distinct from zero is important: zero vegetation is a real observation, while NoData means that the pixel cannot be interpreted.


In [ ]:
# CHANGED — city workflow
ndvi_folder, veg_folder = DATA / 'ndvi-tiles', DATA / 'veg-tiles'
ndvi_folder.mkdir(exist_ok=True)
veg_folder.mkdir(exist_ok=True)
ndvi_tiles, veg_tiles = [], []
for path in processing_tiles:
    with rio.open(path) as src:
        # NAIP band 1 is red and band 4 is near-infrared.
        red = src.read(RED_BAND).astype('float32')
        nir = src.read(NIR_BAND).astype('float32')
        denominator = nir + red
        valid = ((src.read_masks(RED_BAND) > 0) & (src.read_masks(NIR_BAND) > 0)
                 & np.isfinite(denominator) & (denominator > 0))
        # Keep invalid pixels separate from valid NDVI values, including zero.
        ndvi = np.full(red.shape, FLOAT_NODATA, dtype='float32')
        np.divide(nir-red, denominator, out=ndvi, where=valid)
        # Vegetation is a binary interpretation of the valid NDVI pixels.
        veg = np.full(red.shape, VEG_NODATA, dtype='uint8')
        veg[valid] = (ndvi[valid] > NDVI_THRESHOLD).astype('uint8')
        for values, folder, dtype, nodata, manifest in [
            (ndvi, ndvi_folder, 'float32', FLOAT_NODATA, ndvi_tiles),
            (veg, veg_folder, 'uint8', VEG_NODATA, veg_tiles)]:
            output_path = folder / path.name
            profile = src.profile.copy()
            profile.update(count=1, dtype=dtype, nodata=nodata)
            profile.pop('photometric', None)
            with rio.open(output_path, 'w', **profile) as dst:
                dst.write(values, 1)
                dst.update_tags(NDVI_THRESHOLD=str(NDVI_THRESHOLD), RED_BAND=str(RED_BAND), NIR_BAND=str(NIR_BAND))
            manifest.append(output_path)
print('Processed NDVI and vegetation tiles:', len(ndvi_tiles))


## 6. Assemble, aggregate, and map the city products

The processed windows are first written back into complete native-resolution city rasters. They are assembled before downsampling so a coarse cell that crosses a processing-tile edge still averages all of its native pixels.

The shared coarse grid is created once and used for both outputs. Mean NDVI is the average of valid native NDVI pixels. Vegetation fraction is the average of valid native `0/1` vegetation values, so it represents the fraction of observed native pixels classified as vegetation. It is displayed as a percentage by multiplying the fraction by 100.

The coarse boundary mask is reapplied after aggregation. This prevents cells outside Miami from receiving values because of the rectangular raster extent. The coverage diagnostic reports how much of the in-boundary coarse grid has valid imagery; it is not an estimate of the percentage of each cell that is covered.


In [ ]:
# NEW — single native-resolution city outputs
# These processing tiles partition the masked raster with no overlaps.
# Write each tile once into its original grid window; avoid repeated merge reads.
CITY_NDVI_NATIVE = PRODUCTS / 'city_ndvi_native.tif'
CITY_VEG_NATIVE = PRODUCTS / 'city_vegetation_native.tif'
with rio.open(MASKED_FILE) as template:
    native_profile = template.profile.copy()
    native_transform, native_shape = template.transform, template.shape
for paths, target, dtype, nodata in [
    (ndvi_tiles, CITY_NDVI_NATIVE, 'float32', FLOAT_NODATA),
    (veg_tiles, CITY_VEG_NATIVE, 'uint8', VEG_NODATA)]:
    profile = native_profile.copy()
    profile.update(count=1, dtype=dtype, nodata=nodata)
    profile.pop('photometric', None)
    written_pixels = 0
    with rio.open(target, 'w', **profile) as dst:
        for path in paths:
            with rio.open(path) as src:
                col, row = (~native_transform) * (src.transform.c, src.transform.f)
                if not (np.isclose(col, round(col)) and np.isclose(row, round(row))):
                    raise ValueError('Processed tile is not aligned to the native city grid.')
                window = Window(round(col), round(row), src.width, src.height)
                if (src.crs != dst.crs or src.res != dst.res or
                    not src.transform.almost_equals(dst.window_transform(window)) or
                    window.col_off < 0 or window.row_off < 0 or
                    window.col_off + window.width > dst.width or
                    window.row_off + window.height > dst.height):
                    raise ValueError('Processed tile is incompatible with the native city grid.')
                dst.write(src.read(1), 1, window=window)
                written_pixels += src.width * src.height
        if written_pixels != dst.width * dst.height:
            raise ValueError('Incomplete tile manifest: rerun the tile split and processing cells.')
        dst.update_tags(NDVI_THRESHOLD=str(NDVI_THRESHOLD))
    print(target)


In [ ]:
# NEW — shared lower-resolution city grid and on-disk aggregation
if not np.isfinite(CITY_RES_M) or CITY_RES_M <= 0:
    raise ValueError('CITY_RES_M must be positive and finite.')
with rio.open(CITY_NDVI_NATIVE) as src:
    if CITY_RES_M <= max(src.res):
        raise ValueError('Choose CITY_RES_M larger than the native pixel size.')
    # aligned_target gives both products one shared grid and transform.
    city_transform, city_width, city_height = aligned_target(
        src.transform, src.width, src.height, CITY_RES_M)
    city_profile = src.profile.copy()
    city_profile.update(transform=city_transform, width=city_width, height=city_height,
                        dtype='float32', nodata=FLOAT_NODATA)
resolution_label = f'{CITY_RES_M:g}m'
CITY_NDVI = PRODUCTS / f'city_ndvi_{resolution_label}.tif'
CITY_VEG = PRODUCTS / f'city_vegetation_fraction_{resolution_label}.tif'
for source, target, description in [
    (CITY_NDVI_NATIVE, CITY_NDVI, 'Mean native NDVI'),
    (CITY_VEG_NATIVE, CITY_VEG, 'Vegetation fraction among valid native pixels')]:
    with rio.open(source) as src, rio.open(target, 'w', **city_profile) as dst:
        # Average valid native pixels instead of thresholding an averaged NDVI.
        reproject(source=rio.band(src, 1), destination=rio.band(dst, 1),
                  src_nodata=src.nodata, dst_nodata=FLOAT_NODATA,
                  resampling=Resampling.average, warp_mem_limit=128)
        dst.set_band_description(1, description)
        dst.update_tags(NDVI_THRESHOLD=str(NDVI_THRESHOLD), aggregation='average of valid native pixels')
    # The raster is rectangular, so reapply the true city boundary afterward.
    with rio.open(target, 'r+') as dst:
        for _, window in dst.block_windows(1):
            values = dst.read(1, window=window)
            inside = geometry_mask(city_geometries, out_shape=values.shape,
                                   transform=dst.window_transform(window), invert=True)
            values[~inside] = FLOAT_NODATA
            dst.write(values, 1, window=window)
    print(target)


In [ ]:
# NEW — city maps and coverage diagnostic
with rio.open(CITY_NDVI) as a, rio.open(CITY_VEG) as b:
    assert (a.crs, a.transform, a.shape) == (b.crs, b.transform, b.shape)
    mean_ndvi, veg_fraction = a.read(1, masked=True), b.read(1, masked=True)
    extent = plotting_extent(a)
    inside = geometry_mask(city_geometries, out_shape=a.shape, transform=a.transform, invert=True)
    covered = inside & ~np.ma.getmaskarray(mean_ndvi)
    coverage = covered.sum() / max(1, inside.sum())
    print(f'Coarse city cells with data: {coverage:.1%} (not a subpixel coverage estimate)')
    if coverage < 0.99:
        warnings.warn('City coverage is incomplete. Check selected tiles and NoData gaps.')
fig, axes = plt.subplots(1, 2, figsize=(15, 8), constrained_layout=True)
for ax, values, cmap, vmin, vmax, title, label in [
    (axes[0], mean_ndvi, 'RdYlGn', -1, 1, 'Mean NDVI', 'NDVI'),
    (axes[1], 100 * veg_fraction, 'YlGn', 0, 100, 'Vegetation cover', 'Vegetation (%)')]:
    im = ax.imshow(values, extent=extent, origin='upper', cmap=cmap, vmin=vmin, vmax=vmax)
    city_projected.boundary.plot(ax=ax, color='black', linewidth=0.5)
    ax.set_title(f'{CITY_NAME}: {title} ({resolution_label})')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.set_aspect('equal')
    fig.colorbar(im, ax=ax, shrink=0.7, label=label)
map_file = PRODUCTS / f'city_ndvi_vegetation_{resolution_label}.png'
fig.savefig(map_file, dpi=200, bbox_inches='tight')
plt.show()
print('Saved city map:', map_file)


## 7. Summarize results by census tract

The census layer is reprojected to the native city raster CRS before sampling. For each tract, the code masks the native NDVI raster and native binary vegetation raster to the tract polygon, then calculates the mean of valid pixels. The vegetation mean is multiplied by 100 to produce `veg_pct`.

A tract outside the city or without valid imagery receives a missing value. During mapping, all tracts are first drawn in gray, and only the portions intersecting the Miami boundary are overlaid with their NDVI or vegetation value. This makes the boundary explicit and prevents a tract's color from extending beyond the study area.

The final GeoPackage contains the tract attributes and the final PNG contains side-by-side maps for mean NDVI and vegetation percentage.


In [ ]:
# CHANGED — city workflow
TRACT_FILE = INPUT_ROOT / 'data' / 'tract_26917.shp'
tracts = None
if not TRACT_FILE.exists():
    print('Optional census step skipped: supply', TRACT_FILE)
else:
    tracts = gpd.read_file(TRACT_FILE)
    if tracts.crs is None:
        raise ValueError('Census tracts require a CRS.')
    with rio.open(CITY_NDVI_NATIVE) as ndvi_src, rio.open(CITY_VEG_NATIVE) as veg_src:
        # Sampling in the native CRS avoids mixing geographic and projected units.
        tracts = tracts.to_crs(ndvi_src.crs)
        means, percentages = [], []
        for geom in tracts.geometry:
            vals = []
            for src in (ndvi_src, veg_src):
                try:
                    # `filled=False` preserves NoData so it is excluded from means.
                    clipped, _ = mask(src, [mapping(geom)], crop=True, filled=False, indexes=1)
                    vals.append(float(clipped.mean()) if clipped.count() else np.nan)
                except ValueError:
                    vals.append(np.nan)
            means.append(vals[0])
            percentages.append(100 * vals[1])
        tracts['ndvi'] = means
        tracts['veg_pct'] = percentages
    tracts.to_file(PRODUCTS / 'census_tract_summary.gpkg', layer='tracts', driver='GPKG')


In [ ]:
# CHANGED — city workflow
if tracts is not None:
    # Draw all tracts gray first; overlay only geometry inside the city boundary.
    city_tracts = gpd.clip(tracts, city_projected)
    fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
    for ax, column, title in zip(axes, ['ndvi', 'veg_pct'], ['Mean NDVI', 'Vegetation (%)']):
        tracts.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.2,
                    label='Outside Miami boundary')
        city_tracts.plot(column=column, ax=ax, cmap='YlGn', legend=True,
                         missing_kwds={'color': 'lightgrey', 'label': 'No city data'},
                         edgecolor='white', linewidth=0.2)
        city_projected.boundary.plot(ax=ax, color='black', linewidth=0.6)
        left, bottom, right, top = city_projected.total_bounds
        padx, pady = (right-left)*0.03, (top-bottom)*0.03
        ax.set_xlim(left-padx, right+padx)
        ax.set_ylim(bottom-pady, top+pady)
        ax.set_title(f'{CITY_NAME}: census tract {title}')
        ax.set_axis_off()
    fig.savefig(PRODUCTS / 'census_tract_maps.png', dpi=200, bbox_inches='tight')
    plt.show()
